In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

pd.set_option("display.max_colwidth", None)

# List available local input directories to locate the dataset/checkpoints
for dirname, _, filenames in os.walk('datasets'):
    for filename in filenames[:10]:
        print(os.path.join(dirname, filename))


Using device: cuda
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# Full listing of local datasets to find model checkpoint folders too
for dirname, _, filenames in os.walk('datasets'):
    if filenames:
        print(dirname, "->", filenames[:5])

print("\n--- TRAIN ---")
train_df = pd.read_csv('datasets/train.csv')
print(train_df.shape)
print(train_df.columns.tolist())
display(train_df.head())

print("\n--- SAMPLE SUBMISSION ---")
sample_sub = pd.read_csv('datasets/sample_submission.csv')
print(sample_sub.shape)
display(sample_sub.head())


/kaggle/input/competitions/smart-mcq-solver-challenge -> ['sample_submission.csv', 'train.csv', 'test.csv']

--- TRAIN ---
(2000, 8)
['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.,"Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.","Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.","Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.",Martin Heidegger believes that the relationship between time and human existence is cyclical. The past and present are interconnected and the future is predetermined. Human beings do not have free will.,"Martin Heidegger believes that time is an illusion, and the past, present, and future are all happening simultaneously. Humans exist outside of this illusion and are guided by a higher power.",B
1,2,What is accelerator-based light-ion fusion?,"Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.","Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.","Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.","Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 100 kV between the electrodes.","Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fission reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fission can be observed with as little as 10 kV between the electrodes.",A
2,3,Determine the correct option: What is the term used in astrophysics to describe light-matter interactions resulting in energy shifts in the radiation field? among the listed options.,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Martin Heidegger's view on the relationship between time and human existence? carefully.,"Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the f


--- SAMPLE SUBMISSION ---
(500, 2)


,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A B C
3,4,A B C
4,5,A B C


In [3]:
# Check common local model/output locations
import subprocess

print("=== datasets ===")
for dirname, _, filenames in os.walk('datasets'):
    if filenames:
        print(dirname, "->", filenames)

print("\n=== outputs ===")
for dirname, _, filenames in os.walk('outputs'):
    if filenames:
        print(dirname, "->", filenames[:5])


=== /kaggle/input full tree ===
/kaggle/input/competitions/smart-mcq-solver-challenge -> ['sample_submission.csv', 'train.csv', 'test.csv']

=== /kaggle/working ===
/kaggle/working/.virtual_documents -> ['__notebook_source__.ipynb']


In [4]:
from sklearn.model_selection import train_test_split

def build_input_text(row):
    return (
        f"Question: {row['prompt']}\n"
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}\n"
        f"E) {row['E']}"
    )

train_df["input_text"] = train_df.apply(build_input_text, axis=1)

label2id = {"A":0, "B":1, "C":2, "D":3, "E":4}
id2label = {v:k for k,v in label2id.items()}
train_df["label"] = train_df["answer"].map(label2id)

train_data, val_data = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df["label"]
)
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print(train_data.shape, val_data.shape)
val_data.head()

(1600, 10) (400, 10)


,id,prompt,A,B,C,D,E,answer,input_text,label
0,1284,What is a phageome?,"A community of viruses and their metagenomes localized in a particular environment, similar to a microbiome.","A community of bacteria and their metagenomes localized in a particular environment, similar to a microbiome.","A community of bacteriophages and their metagenomes localized in a particular environment, similar to a microbiome.","A community of fungi and their metagenomes localized in a particular environment, similar to a microbiome.","A community of archaea and their metagenomes localized in a particular environment, similar to a microbiome.",C,"Question: What is a phageome?\nA) A community of viruses and their metagenomes localized in a particular environment, similar to a microbiome.\nB) A community of bacteria and their metagenomes localized in a particular environment, similar to a microbiome.\nC) A community of bacteriophages and their metagenomes localized in a particular environment, similar to a microbiome.\nD) A community of fungi and their metagenomes localized in a particular environment, similar to a microbiome.\nE) A community of archaea and their metagenomes localized in a particular environment, similar to a microbiome.",2
1,88,Identify the correct statement: What is the role of CYCLOIDEA genes in the evolution of bilateral symmetry? carefully.,CYCLOIDEA genes are responsible for the selection of symmetry in the evolution of animals.,"CYCLOIDEA genes are responsible for the evolution of specialized pollinators in plants, which in turn led to the transition of radially symmetrical flowers to bilaterally symmetrical flowers.","CYCLOIDEA genes are responsible for the expression of dorsal petals in Antirrhinum majus, which control their size and shape.","CYCLOIDEA genes are responsible for the expression of transcription factors that control the expression of other genes, allowing their expression to influence developmental pathways relating to symmetry.",CYCLOIDEA genes are responsible for mutations that cause a reversion to radial symmetry.,D,"Question: Identify the correct statement: What is the role of CYCLOIDEA genes in the evolution of bilateral symmetry? carefully.\nA) CYCLOIDEA genes are responsible for the selection of symmetry in the evolution of animals.\nB) CYCLOIDEA genes are responsible for the evolution of specialized pollinators in plants, which in turn led to the transition of radially symmetrical flowers to bilaterally symmetrical flowers.\nC) CYCLOIDEA genes are responsible for the expression of dorsal petals in Antirrhinum majus, which control their size and shape.\nD) CYCLOIDEA genes are responsible for the expression of transcription factors that control the expression of other genes, allowing their expression to influence developmental pathways relating to symmetry.\nE) CYCLOIDEA genes are responsible for mutations that cause a reversion to radial symmetry.",3
2,1213,Determine the correct option: What did Fresnel predict and verify with regards to total internal reflections?,"Fresnel predicted and verified that three total internal reflections at 75°27' would give a precise circular polarization if two of the reflections had water as the external medium and the third had air, but not if the reflecting surfaces were all wet or all dry.","Fresnel predicted and verified that eight total internal reflections at 68°27' would give an accurate circular polarization if four of the reflections had water as the external medium while the other four had air, but not if the reflecting surfaces were all wet or all dry.","Fresnel predicted and verified that four total internal reflections at 30°27' would result in circular polarization if two of the reflections had water as the external medium while the other two had air, regardless if the reflecting surfaces were all wet or all dry.","Fresnel predicted and verified that two total internal reflections at 68°27' would give an accurate linear polarization if one of the reflecti

In [5]:
from torch.utils.data import Dataset

MAX_LEN = 384  # combined prompt+options can be long; truncate safely

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.texts = df["input_text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Tokenizers for both models
deberta_ckpt = "microsoft/deberta-v3-small"
roberta_ckpt = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_ckpt)
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_ckpt)

print("Tokenizers loaded.")
print(deberta_tokenizer.tokenize(train_data.iloc[0]["input_text"])[:20])

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizers loaded.
['▁Question', ':', '▁Select', '▁the', '▁most', '▁accurate', '▁option', ':', '▁What', '▁is', '▁Modified', '▁Newtonian', '▁Dynamics', '▁(', 'MOND', ')', '?', '▁from', '▁the', '▁following']


In [6]:
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from torch.optim import AdamW
import time

def train_model(model_ckpt, tokenizer, train_data, val_data, num_epochs=3, batch_size=8, lr=2e-5):
    model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=5)
    model.to(device)

    train_ds = MCQDataset(train_data, tokenizer)
    val_ds   = MCQDataset(val_data,   tokenizer)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size*2, shuffle=False, num_workers=2)

    optimizer = AdamW(model.parameters(), lr=lr, eps=1e-6, weight_decay=0.01)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    # use bf16 if available, else fp32
    use_bf16 = torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float32
    print(f"Using AMP dtype: {amp_dtype}")

    for epoch in range(num_epochs):
        model.train()
        start = time.time()
        total_loss, n_batches = 0, 0

        for batch in train_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            optimizer.zero_grad()

            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits  = outputs.logits  # [B, 5]
                loss    = torch.nn.functional.cross_entropy(logits, labels)

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            n_batches  += 1

        avg_loss = total_loss / max(n_batches, 1)

        # validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels         = batch["labels"].to(device)
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=-1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)
        val_acc = correct / total

        print(f"Epoch {epoch+1}/{num_epochs} | loss={avg_loss:.4f} | val_acc={val_acc:.4f} | time={time.time()-start:.1f}s")

    return model

print("Training DeBERTa...")
deberta_model = train_model(deberta_ckpt, deberta_tokenizer, train_data, val_data, num_epochs=3, batch_size=8, lr=2e-5)

Training DeBERTa...


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

Using AMP dtype: torch.bfloat16
Epoch 1/3 | loss=1.6773 | val_acc=0.2300 | time=105.5s
Epoch 2/3 | loss=1.6110 | val_acc=0.2450 | time=106.2s
Epoch 3/3 | loss=1.6012 | val_acc=0.2300 | time=106.4s


In [7]:
print("Training RoBERTa...")
roberta_model = train_model(roberta_ckpt, roberta_tokenizer, train_data, val_data, num_epochs=3, batch_size=8, lr=2e-5)

Training RoBERTa...


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using AMP dtype: torch.bfloat16
Epoch 1/3 | loss=1.3673 | val_acc=0.8350 | time=169.2s
Epoch 2/3 | loss=0.1460 | val_acc=1.0000 | time=169.1s
Epoch 3/3 | loss=0.0044 | val_acc=1.0000 | time=168.8s


In [8]:
# Save both models immediately
deberta_model.save_pretrained('outputs/deberta_finetuned')
deberta_tokenizer.save_pretrained('outputs/deberta_finetuned')

roberta_model.save_pretrained('outputs/roberta_finetuned')
roberta_tokenizer.save_pretrained('outputs/roberta_finetuned')
print("Both models saved!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Both models saved!


In [9]:
# Load test data and run inference on row index 25
test_df = pd.read_csv('datasets/test.csv')
test_df["input_text"] = test_df.apply(build_input_text, axis=1)

id2label = {0:"A", 1:"B", 2:"C", 3:"D", 4:"E"}

def get_probs(model, tokenizer, text):
    model.eval()
    enc = tokenizer(
        text,
        truncation=True,
        max_length=384,
        padding="max_length",
        return_tensors="pt"
    )
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
    return probs

sample_text = test_df.iloc[25]["input_text"]
print("Row 25 prompt preview:", test_df.iloc[25]["prompt"][:80])
print()

# Q1: DeBERTa probabilities
deb_probs = get_probs(deberta_model, deberta_tokenizer, sample_text)
print("DeBERTa probs:", {id2label[i]: round(float(deb_probs[i]), 4) for i in range(5)})
deb_top = int(deb_probs.argmax())
print(f"Q1: {id2label[deb_top]}, {deb_probs[deb_top]:.4f}")
print()

# Q2: RoBERTa probabilities + simple average ensemble
rob_probs = get_probs(roberta_model, roberta_tokenizer, sample_text)
print("RoBERTa probs:", {id2label[i]: round(float(rob_probs[i]), 4) for i in range(5)})
avg_probs = (deb_probs + rob_probs) / 2
avg_top = int(avg_probs.argmax())
print()
print("Avg probs:", {id2label[i]: round(float(avg_probs[i]), 4) for i in range(5)})
print(f"Q2: {id2label[avg_top]}")


Row 25 prompt preview: Determine the correct option: What is the most popular explanation for the showe

DeBERTa probs: {'A': 0.1826, 'B': 0.2354, 'C': 0.2468, 'D': 0.1779, 'E': 0.1573}
Q1: C, 0.2468

RoBERTa probs: {'A': 0.0007, 'B': 0.0008, 'C': 0.0006, 'D': 0.0006, 'E': 0.9973}

Avg probs: {'A': 0.0917, 'B': 0.1181, 'C': 0.1237, 'D': 0.0892, 'E': 0.5773}
Q2: E


In [10]:
# Q3: Weighted ensemble (DeBERTa 0.7, RoBERTa 0.3)
weighted_probs = 0.7 * deb_probs + 0.3 * rob_probs
weighted_top = int(weighted_probs.argmax())
print("Weighted probs:", {id2label[i]: round(float(weighted_probs[i]), 4) for i in range(5)})
print(f"Q3: {id2label[weighted_top]}")
print()

# Q4: Top-3 prediction string for row index 25
top3_indices = weighted_probs.argsort()[::-1][:3]
top3_str = " ".join([id2label[i] for i in top3_indices])
print(f"Q4: {top3_str}")

Weighted probs: {'A': 0.128, 'B': 0.165, 'C': 0.173, 'D': 0.1247, 'E': 0.4094}
Q3: E

Q4: E C B


In [11]:
# Q5: Run weighted ensemble on ALL test rows, save submission.csv
from tqdm import tqdm

def get_probs_batch(model, tokenizer, texts, batch_size=16):
    all_probs = []
    model.eval()
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(
            batch_texts,
            truncation=True,
            max_length=384,
            padding="max_length",
            return_tensors="pt"
        )
        input_ids      = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)

texts = test_df["input_text"].tolist()

print("Running DeBERTa on full test set...")
deb_all = get_probs_batch(deberta_model, deberta_tokenizer, texts)

print("Running RoBERTa on full test set...")
rob_all = get_probs_batch(roberta_model, roberta_tokenizer, texts)

# Weighted ensemble
weighted_all = 0.7 * deb_all + 0.3 * rob_all

# Build top-3 predictions
predictions = []
for i in range(len(test_df)):
    top3 = weighted_all[i].argsort()[::-1][:3]
    top3_str = " ".join([id2label[j] for j in top3])
    predictions.append(top3_str)

# Save submission
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})
submission.to_csv('outputs/submission.csv', index=False)

print(f"\nQ5: {len(submission)} prediction rows")
print(submission.head())


Running DeBERTa on full test set...


100%|██████████| 32/32 [00:03<00:00,  9.62it/s]


Running RoBERTa on full test set...


100%|██████████| 32/32 [00:14<00:00,  2.24it/s]


Q5: 500 prediction rows
   id prediction
0   1      A C B
1   2      B C A
2   3      B C A
3   4      E C B
4   5      C B A


In [12]:
# Q6: TTA on first 50 rows of test.csv
# Two versions: original prompt vs instruction-augmented prompt

def build_augmented_input(row):
    return (
        f"Answer the following multiple-choice question carefully:\n"
        f"Question: {row['prompt']}\n"
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}\n"
        f"E) {row['E']}"
    )

test_50 = test_df.iloc[:50].reset_index(drop=True)

# Original prompts
orig_texts = test_50["input_text"].tolist()
# Augmented prompts
aug_texts = [build_augmented_input(row) for _, row in test_50.iterrows()]

print("Running DeBERTa on original prompts...")
deb_orig = get_probs_batch(deberta_model, deberta_tokenizer, orig_texts, batch_size=16)

print("Running DeBERTa on augmented prompts...")
deb_aug = get_probs_batch(deberta_model, deberta_tokenizer, aug_texts, batch_size=16)

# TTA: average both passes
tta_probs = (deb_orig + deb_aug) / 2

# Compare Top-1 predictions
orig_top1 = deb_orig.argmax(axis=1)
tta_top1  = tta_probs.argmax(axis=1)

different = (orig_top1 != tta_top1).sum()
print(f"\nQ6: {different} rows have different Top-1 prediction after TTA")

Running DeBERTa on original prompts...


100%|██████████| 4/4 [00:00<00:00, 10.74it/s]


Running DeBERTa on augmented prompts...


100%|██████████| 4/4 [00:00<00:00, 12.51it/s]


Q6: 0 rows have different Top-1 prediction after TTA


In [13]:
# Q7, Q8, Q9: First 100 rows of test.csv
test_100 = test_df.iloc[:100].reset_index(drop=True)
texts_100 = test_100["input_text"].tolist()

# Already have deb_all and weighted_all from Q5, just slice first 100
deb_100      = deb_all[:100]
weighted_100 = weighted_all[:100]

# Q7: How many rows have different Top-1 between DeBERTa and Weighted Ensemble
deb_top1      = deb_100.argmax(axis=1)
weighted_top1 = weighted_100.argmax(axis=1)
q7 = (deb_top1 != weighted_top1).sum()
print(f"Q7: {q7} rows have different Top-1 predictions")

# Q8: Confidence gain - how many rows have positive confidence gain
deb_conf      = deb_100.max(axis=1)
weighted_conf = weighted_100.max(axis=1)
conf_gain     = weighted_conf - deb_conf
q8 = (conf_gain > 0).sum()
print(f"Q8: {q8} rows have positive confidence gain")

# Q9: How many rows have at least one change in Top-3 ordering
deb_top3      = deb_100.argsort(axis=1)[:, ::-1][:, :3]
weighted_top3 = weighted_100.argsort(axis=1)[:, ::-1][:, :3]

q9 = 0
for i in range(100):
    deb_str = " ".join([id2label[j] for j in deb_top3[i]])
    ens_str = " ".join([id2label[j] for j in weighted_top3[i]])
    if deb_str != ens_str:
        q9 += 1
print(f"Q9: {q9} rows have at least one change in Top-3 ranking")

Q7: 74 rows have different Top-1 predictions
Q8: 100 rows have positive confidence gain
Q9: 74 rows have at least one change in Top-3 ranking


In [14]:
# Q10: MAP@3 on first 100 validation samples (val_data has ground truth labels)
val_100 = val_data.iloc[:100].reset_index(drop=True)
val_100["input_text"] = val_100.apply(build_input_text, axis=1)
val_texts = val_100["input_text"].tolist()

print("Running weighted ensemble on first 100 val samples...")
deb_val = get_probs_batch(deberta_model, deberta_tokenizer, val_texts, batch_size=16)
rob_val = get_probs_batch(roberta_model, roberta_tokenizer, val_texts, batch_size=16)
weighted_val = 0.7 * deb_val + 0.3 * rob_val

# Compute MAP@3
def mapk(actual, predicted, k=3):
    score = 0.0
    for a, p in zip(actual, predicted):
        hits = 0
        for i, pred in enumerate(p[:k]):
            if pred == a:
                hits += 1
                score += hits / (i + 1)
                break
    return score / len(actual)

# Get top-3 predictions and true labels
top3_preds = []
for i in range(100):
    top3 = weighted_val[i].argsort()[::-1][:3]
    top3_preds.append([id2label[j] for j in top3])

true_labels = [id2label[l] for l in val_100["label"].tolist()]

map3 = mapk(true_labels, top3_preds, k=3)
print(f"\nQ10: MAP@3 = {map3:.4f}")

Running weighted ensemble on first 100 val samples...


100%|██████████| 7/7 [00:02<00:00,  2.80it/s]


Q10: MAP@3 = 1.0000


In [15]:
print("Retraining RoBERTa with 1 epoch...")
roberta_model_1ep = train_model(roberta_ckpt, roberta_tokenizer, train_data, val_data, num_epochs=1, batch_size=8, lr=2e-5)

Retraining RoBERTa with 1 epoch...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using AMP dtype: torch.bfloat16
Epoch 1/1 | loss=1.3331 | val_acc=0.7700 | time=170.3s


In [16]:
print("Running weighted ensemble on first 100 val samples (1-epoch RoBERTa)...")
deb_val = get_probs_batch(deberta_model, deberta_tokenizer, val_texts, batch_size=16)
rob_val = get_probs_batch(roberta_model_1ep, roberta_tokenizer, val_texts, batch_size=16)
weighted_val = 0.7 * deb_val + 0.3 * rob_val

top3_preds = []
for i in range(100):
    top3 = weighted_val[i].argsort()[::-1][:3]
    top3_preds.append([id2label[j] for j in top3])

map3 = mapk(true_labels, top3_preds, k=3)
print(f"\nQ10: MAP@3 = {map3:.4f}")

Running weighted ensemble on first 100 val samples (1-epoch RoBERTa)...


100%|██████████| 7/7 [00:02<00:00,  3.01it/s]


Q10: MAP@3 = 0.7767
